<a href="https://colab.research.google.com/github/saigireesh-Png/ShopShpere-Ecom/blob/main/Another_copy_of_MIddleware.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U "langchain>=1.3.14"

In [ ]:
!pip install  langchain-openai langchain-community langgraph python-dotenv langchain-mcp-adapters langchain-chroma chromadb pypdf

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

False

In [ ]:
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

SecretNotFoundError: Secret OPENAI_API_KEY does not exist.

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model('openai:gpt-5-mini')
model.invoke('Hi')
print("Cinebot's Brain is connected")

In [ ]:
print("hello")

In [ ]:
from langchain_core.tools import tool
import sqlite3


In [ ]:

@tool
def save_trip_demo(user_id: str, destination: str) -> str:
    """Save a trip to the database. Irreversible without manual cleanup."""
    return f"Trip to {destination} saved for {user_id}."  # standing in for a real DB write

In [ ]:
from langchain.agents import create_agent

from langchain.agents.middleware import SummarizationMiddleware


agent = create_agent(
    model="openai:gpt-5-mini",
    tools=[save_trip_demo],
    middleware=[
        SummarizationMiddleware(
            model = 'gpt-5.4-mini',
            trigger = ('token',4000),
            keep =('messages',10),
        )
    ])

Summarization is text-oriented context compression. It does not resize, downsample, or otherwise compress image/audio/video payloads. Recent messages retained by keep still include their original multimodal blocks, while older multimodal messages that are summarized are represented only by the generated text summary. For image-heavy applications, store media in a filesystem or object store and pass URLs or file references through message history.

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver


def your_read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def your_send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

agent = create_agent(
    model="gpt-5.5",
    tools=[your_read_email_tool, your_send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "your_send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
                "your_read_email_tool": False,
            }
        ),
    ],
)

In [ ]:
config = {'configurable':{"thread_id":"hitl"}}

result =agent.invoke({"messages": [("user", "Send an email to my manager on mayank953ai@gmail.com, asking for a leave")]}, config=config)

In [ ]:
result

```
User
  │
  │ "Send an email asking for leave"
  ▼
LLM
  │
  │ decides to call
  ▼
your_send_email_tool
  │
  │ requires approval
  ▼
INTERRUPT
  │
  ├── approve → send email
  ├── edit    → modify tool arguments
  └── reject  → don't send
```

In [ ]:
# --- Core LangChain ---
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain.tools import tool as tool_rt, ToolRuntime

In [ ]:
# --- LangGraph (checkpointing, resuming) ---
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

In [ ]:
from langchain.agents.middleware import (
    SummarizationMiddleware,
    HumanInTheLoopMiddleware,
    ModelCallLimitMiddleware,
    ToolCallLimitMiddleware,
    ModelFallbackMiddleware,
    PIIMiddleware,
    TodoListMiddleware,
    LLMToolSelectorMiddleware,
    ToolRetryMiddleware,
    ModelRetryMiddleware,
    LLMToolEmulator,
    ContextEditingMiddleware,
    ClearToolUsesEdit,
)

# Please start from here

Defining Tools for my Cinebot - Dummy as of now but ofcourse will be real as we have seen in last class.

In [ ]:
@tool
def check_showtimes(movie_title: str) -> str:
    """Check available showtimes for a movie at the cinema."""
    fake_showtimes = {
        "interstellar": "7:00 PM and 10:15 PM",
        "dune part two": "9:30 PM only",
        "oppenheimer": "Sold out for tonight",
    }
    return fake_showtimes.get(movie_title.lower(), "No showtimes found for that title.")



In [ ]:

@tool
def book_seats(movie_title: str, seat_count: int) -> str:
    """Book seats for a movie. Irreversible once confirmed."""
    return f"Booked {seat_count} seat(s) for {movie_title}."


In [ ]:
@tool
def cancel_booking(booking_id: str) -> str:
    """Cancel an existing booking. Irreversible."""
    return f"Booking {booking_id} cancelled."


In [ ]:
@tool
def check_order_status(booking_id: str) -> str:
    """Check the status of an existing booking."""
    return f"Booking {booking_id}: confirmed, 2 seats, Interstellar, 7:00 PM."


In [ ]:
@tool
def get_refund_policy() -> str:
    """Get the cinema's refund policy -- exact wording, not to be paraphrased."""
    return "Refunds available up to 2 hours before showtime. No refunds after that."

In [ ]:

@tool
def lookup_seat_map(movie_title: str, seat_number: str) -> str:
    """Look up a specific seat -- fails if the seat number format is wrong."""
    if not seat_number or not seat_number[0].isalpha():
        raise ValueError(f"Malformed seat number '{seat_number}' -- expected a letter+number like 'A12'.")
    return f"Seat {seat_number} for {movie_title}: available."


In [ ]:
cinebot_tools = [check_showtimes, book_seats, cancel_booking, check_order_status, get_refund_policy, lookup_seat_map]


In [ ]:
def pretty_print_messages(result):
    for i, message in enumerate(result.get("messages", []), 1):
        print(f"\n{'=' * 80}")
        print(f"Message {i}: {message.__class__.__name__}")
        print("=" * 80)

        # Basic message information
        print(f"ID: {getattr(message, 'id', None)}")

        # Message content
        content = getattr(message, "content", "")
        if content:
            print("\nContent:")
            print(content)

        # Tool calls
        tool_calls = getattr(message, "tool_calls", None)
        if tool_calls:
            print("\nTool Calls:")
            for tool in tool_calls:
                print(f"  • {tool['name']}")
                print(f"    Args: {tool['args']}")
                print(f"    ID:   {tool['id']}")

        # Tool message information
        tool_call_id = getattr(message, "tool_call_id", None)
        if tool_call_id:
            print(f"\nTool Call ID: {tool_call_id}")

        # Summarization information
        additional_kwargs = getattr(message, "additional_kwargs", {})
        if additional_kwargs.get("lc_source"):
            print(f"\nSource: {additional_kwargs['lc_source']}")

    print(f"\n{'=' * 80}")
    print("END OF MESSAGE HISTORY")
    print("=" * 80)

# Summarization Middleware

In [ ]:
plz add comment for token if > this summarise previous messages and keep last 3 messages only . help us to memorise it

In [ ]:
summarizing_agent = create_agent(
    model="gpt-5.4-mini",
    tools=cinebot_tools,
    middleware=[
        SummarizationMiddleware(
            model="gpt-5.4-mini",
            trigger=("tokens", 300),
            keep=("messages", 1),
        )
    ],
)



In [ ]:
print(summarizing_agent.invoke({"messages": [("user", "Hi I am mayank ")]}))

In [ ]:
print(summarizing_agent.invoke({"messages": [("user", "Who am I ? ")]}))

In [ ]:
result = summarizing_agent.invoke({"messages": [("user", "Is Interstellar showing tonight? also please make sure that you book me a ticket, refund me if it is not available,also share the refund policy for me to go through, also check my order status for book_1234")]})

In [ ]:
pretty_print_messages(result)

# HITL (Human in the Loop)

In [ ]:
guarded_agent = create_agent(
    model="openai:gpt-5-mini",
    tools=cinebot_tools,
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={"cancel_booking": {"allowed_decisions": ["approve", "edit", "reject", "respond"]}}
        ),
    ],
    checkpointer=InMemorySaver(),  # REQUIRED -- HITL needs to pause and later resume
)

config = {'configurable':{'thread_id':'hitl-demo-live'}}

In [ ]:
print(guarded_agent.invoke({"messages": [("user", "I am Mayank . ")]}),config=config)

In [ ]:
print(summarizing_agent.invoke({"messages": [("user", "Who am I ? ")]}))

In [ ]:
result = guarded_agent.invoke({"messages": [("user", "Please cancel booking BK1042")]}, config=config)

In [ ]:
from rich import print

In [ ]:
print(result)

In [ ]:
state = guarded_agent.get_state(config)

In [ ]:
state # Please paste it into Chat GPT and ask it to explain the same or Mayank GPT

In [ ]:
print(state.tasks[0])

In [ ]:
resumed_result = guarded_agent.invoke(Command(resume={"decisions":[{"type":"approve"}]}),config=config)

In [ ]:
print(resumed_result)

In [ ]:
state.next

In [ ]:
def run_interactive_hitl_demo(agent, config):
    """A genuinely interactive HITL loop -- ask out loud, type the answer, watch it apply live."""
    state = agent.get_state(config)
    if not state.next:
        print("Nothing is currently paused for approval.")
        return

    print("The agent wants to call a guarded tool. Choose a decision:")
    print("  1) approve  -- run it exactly as proposed")
    print("  2) edit     -- run it, but change the booking_id first")
    print("  3) reject   -- block it, with a reason sent back to the agent")
    print("  4) respond  -- answer a question instead of deciding on the action")

    choice = input("Type 1, 2, 3, or 4: ").strip()

    if choice == "1":
        decision = {"type": "approve"}
    elif choice == "2":
        new_id = input("New booking_id to use instead: ").strip()
        decision = {"type": "edit", "args": {"booking_id": new_id}}
    elif choice == "3":
        reason = input("Reason for rejecting: ").strip()
        decision = {"type": "reject", "message": reason}
    elif choice == "4":
        answer = input("Your response to the agent: ").strip()
        decision = {"type": "respond", "message": answer}
    else:
        print("Not a valid choice -- try again.")
        return

    resumed = agent.invoke(Command(resume={"decisions": [decision]}), config=config)
    print()
    print("Agent's final response:", resumed["messages"][-1].content)


In [ ]:
config = {'configurable':{'thread_id':'hitl-demo-live-2'}}

In [ ]:
guarded_agent = create_agent(
    model="openai:gpt-5-mini",
    tools=cinebot_tools,
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={"cancel_booking": {"allowed_decisions": ["approve", "edit", "reject", "respond"]}}
        ),
    ],
    checkpointer=InMemorySaver(),  # REQUIRED -- HITL needs to pause and later resume
)


In [ ]:
result = guarded_agent.invoke({"messages": [("user", "Please cancel booking BK1042")]}, config=config)

In [ ]:
run_interactive_hitl_demo(guarded_agent, config)

> ⚙️ **Code Walkthrough:** `Command(resume={"decisions": [...]})` is how you hand a decision
> back to an agent that's paused mid-run. The `decisions` list has one entry per interrupted
> tool call (usually just one). Each decision is a dict with a `"type"` key matching one of the
> four options, plus whatever extra data that type needs — `edit` needs new `args`, `reject` and
> `respond` need a `message`, `approve` needs nothing else.

# Model Call Limit

In [ ]:
call_limited_agent = create_agent(
    model="openai:gpt-5-mini",
    tools=cinebot_tools,
    checkpointer=InMemorySaver(),  # required for thread_limit to persist across calls
    middleware=[
        ModelCallLimitMiddleware( # low values taken just for example
            thread_limit=5,   # across the WHOLE conversation
            run_limit=2,       # per single .invoke() call
            exit_behavior="end",  # graceful stop, not an exception
        ),
    ],
)

In [ ]:
result = call_limited_agent.invoke(
    {"messages": [("user", "Can you tell me cinema's refund policy? ")]},
    config={"configurable": {"thread_id": "call-limit-demo-4"}},
)

In [ ]:
print(result)

In [ ]:
result_invoke_2= call_limited_agent.invoke(
    {"messages": [("user", "cancel my booking B123? ")]},
    config={"configurable": {"thread_id": "call-limit-demo-4"}},
)

In [ ]:
result_invoke3 = call_limited_agent.invoke(
    {"messages": [("user", "What all movies are being shown? ")]},
    config={"configurable": {"thread_id": "call-limit-demo-4"}},
)

In [ ]:
print(result_invoke3)

In [ ]:
result_invoke4 = call_limited_agent.invoke(
    {"messages": [("user", "Summarize my chat? ")]},
    config={"configurable": {"thread_id": "call-limit-demo-4"}},
)

In [ ]:
print(result_invoke4)

In [ ]:
from rich import print

# Model Fallback

In [ ]:
resilient_agent = create_agent(
    model="openai:gpt-5.5-haiku",     # primary, most capable
    tools=cinebot_tools,
)

In [ ]:
result = resilient_agent.invoke( {"messages": [("user", "Summarize my chat? ")]},)

In [ ]:
resilient_agent = create_agent(
    model="openai:gpt-5.5-haiku",     # primary, most capable
    tools=cinebot_tools,
    middleware=[
        ModelFallbackMiddleware(
            "openai:gpt-5-mini",   # fallback -- cheaper, still OpenAI, needs no extra setup
            # "ollama:llama3.2",   # a further, fully-local last resort -- uncomment if you have
                                    # `pip install langchain-ollama` AND a local Ollama server running.
                                    # Left commented here so this cell runs with nothing beyond
                                    # what Setup already installed.
        ),
    ],
)
print("Fallback chain: gpt-5.5 -> gpt-5-mini.")
print("If the primary model call fails for any reason, this silently tries the next one.")

In [ ]:
result = resilient_agent.invoke( {"messages": [("user", "Summarize my chat? ")]},)

In [ ]:
from IPython.display import Image, display

graph = resilient_agent.get_graph()

display(Image(graph.draw_mermaid_png()))

In [ ]:
print(result)

# Tool Call Limit

In [ ]:
tool_limited_agent = create_agent(
    model="openai:gpt-5-mini",
    tools=cinebot_tools,
    checkpointer=InMemorySaver(),
    middleware=[
        ToolCallLimitMiddleware(run_limit=8),                              # global, this turn
        ToolCallLimitMiddleware(tool_name="cancel_booking", thread_limit=2),  # tighter, one tool, whole conversation
    ],
)

**Fraud detection in fintech:** a `wrap_tool_call`-style hook around every transaction-executing
tool, flagging or blocking based on velocity, amount, or pattern — exactly `CineBotComplianceMiddleware`'s
rate-limiting logic, aimed at fraud instead of booking abuse.

**Healthcare AI assistants:** `PIIMiddleware`-style redaction is often a genuine *legal*
requirement (HIPAA in the US) — the pattern from Part 2's PII sections isn't optional polish in
that industry, it's the difference between compliant and non-compliant software.

**Customer support platforms:** `HumanInTheLoopMiddleware`-style approval gates before any action
with real financial consequence (refunds, account changes) — the exact shape used to guard
`cancel_booking` here, used to guard a real refund tool there.

**Internal developer tools:** audit logging via `wrap_tool_call`, like Part 3's
`BookingAuditMiddleware`, is close to universal in any system where "who did what, when" needs a
real answer — compliance, debugging, or just accountability.

# 9th August

In [ ]:
tool_limited_agent = create_agent(
    model="openai:gpt-5-mini",
    tools=cinebot_tools,
    checkpointer=InMemorySaver(),
    middleware=[
        ToolCallLimitMiddleware(run_limit=8),                              # global, this turn
        ToolCallLimitMiddleware(tool_name="cancel_booking", thread_limit=2, run_limit=1),  # tighter, one tool, whole conversation
    ],
)

In [ ]:
config = {"configurable": {"thread_id": "tool-limit-demo"}}


In [ ]:
for i in range(3):
  result = tool_limited_agent.invoke({"messages": [("user", f"Please cancel my Booking with ID B{100+i} ? ")]}, config=config)
  print(result)

# PII Detection

In [ ]:
pii_agent = create_agent(
    model="openai:gpt-5-mini",
    tools=cinebot_tools,
    middleware=[
        PIIMiddleware("email", strategy="redact", apply_to_input=True),
        PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),
    ],
)



In [ ]:
result = pii_agent.invoke({
    "messages": [("user", "My email is priya@example.com and my credit card is 4111-1111-1111-1234, can you check showtimes for Dune?")]
})

In [ ]:
print(result['messages'][-1].content)

In [ ]:
print(result)

In [ ]:
# BK1101

In [ ]:
import re

In [ ]:
#https://reference.langchain.com/python/langchain/agents/middleware/pii/PIIMiddleware

In [ ]:

def detect_booking_code(content: str) -> list[dict]:
    """Detect CineBot's own booking code format: BK followed by 4 digits."""
    matches = []
    for match in re.finditer(r"BK\d{4}", content):
        matches.append({"text": match.group(0), "start": match.start(), "end": match.end()})
    return matches


In [ ]:
custom_pii_agent = create_agent(
    model="openai:gpt-5-mini",
    tools=cinebot_tools,
    middleware=[PIIMiddleware("booking_code", detector=detect_booking_code, strategy="mask")],
)

In [ ]:
result = custom_pii_agent.invoke({
    "messages": [("user", "Can you check the status of my booking BK1044 for me?")]
})

In [ ]:
print(result)

# TO Do List

In [ ]:
todo_agent = create_agent(
    model="openai:gpt-5-mini",
    tools=cinebot_tools,
    middleware=[TodoListMiddleware()],
    system_prompt=
)

In [ ]:
result = todo_agent.invoke({
    "messages": [("user", "I want to plan a movie night: check what's showing, pick something good, and book 2 seats.")]
})

In [ ]:
print(result)

# LLM Tool Selector

In [ ]:
for tool in cinebot_tools:
  print (tool.name)

In [ ]:
from langchain.agents.middleware import wrap_model_call

@wrap_model_call
def show_tools(request, handler):
    print("\nTOOLS SENT TO MODEL:")
    print([tool.name for tool in request.tools])

    return handler(request)

In [ ]:
selector_agent = create_agent(
    model="openai:gpt-5-mini",
    tools=cinebot_tools,
    middleware=[
        LLMToolSelectorMiddleware(
            model="openai:gpt-5-mini",     # can be a CHEAPER model than the main agent
            max_tools=2,
            always_include=["check_showtimes"],  # always kept, doesn't count against max_tools
        ),
        show_tools
    ],
)

In [ ]:
result = selector_agent.invoke({"messages": [("user", "Can you cancel my booking with ID B1234?")]})

In [ ]:
print(result)

# Tool Error

In [ ]:
from rich import print

In [ ]:
import langchain

print(langchain.__version__)

In [ ]:
!pip install -U "langchain>=1.3.14"

In [ ]:
from langchain.agents.middleware import ToolErrorMiddleware


In [ ]:
@tool
def lookup_seat_map(movie_title: str, seat_number: str) -> str:
    """Look up a specific seat -- fails if the seat number format is wrong."""
    if not seat_number or not seat_number[0].isalpha():
        raise ValueError(f"Malformed seat number '{seat_number}' -- expected a letter+number like 'A12'.")
    return f"Seat {seat_number} for {movie_title}: available."

In [ ]:
def on_seat_error(exc: Exception, request) -> str | None:
    if isinstance(exc, ValueError):
        # Return the EXCEPTION TYPE, not str(exc) -- internal detail never reaches the model
        return f"`{request.tool_call['name']}` failed with {type(exc).__name__}. Please provide a valid seat number like 'A12'."
    return None  # anything else propagates and halts the run


In [ ]:
error_handled_agent = create_agent(
    model="openai:gpt-5-mini",
    tools=cinebot_tools,
    # middleware=[ToolErrorMiddleware(on_error=on_seat_error)],
)

In [ ]:
result = error_handled_agent.invoke({"messages": [("user", "Look up seat 12 for Dune Part Two")]})

In [ ]:
error_handled_agent = create_agent(
    model="openai:gpt-5-mini",
    tools=cinebot_tools,
    middleware=[ToolErrorMiddleware(on_error=on_seat_error)],
)

In [ ]:
result = error_handled_agent.invoke({"messages": [("user", "Look up seat 12 for Dune Part Two")]})

In [ ]:
print(result)

# Tool Rety

In [ ]:
from langchain.agents.middleware import ToolRetryMiddleware

import random
random.random()



In [ ]:
import random
@tool
def flaky_showtime_check(movie_title: str) -> str:
    """Check showtimes via an external service that can transiently fail."""
    if not random.random() > 1:
        print("Facing Connection Error")
        raise ConnectionError("Simulated network failure -- exactly what a real external call risks.")
    return f"{movie_title}: showing at 8:00 PM."


In [ ]:

resilient_tool_agent = create_agent(
    model="openai:gpt-5-mini",
    tools=[flaky_showtime_check],
    middleware=[
        ToolRetryMiddleware(max_retries=3, backoff_factor=2.0, initial_delay=1.0, on_failure="continue"),
    ],

)

In [ ]:
result = resilient_tool_agent.invoke({"messages": [("user", "Check showtimes for Interstellar")]})


In [ ]:
print(result)

In [ ]:
initial_delay=1.0
backoff_factor=2.0